# Notebook 14 — Censored-MLE AID 1851 population moments, and the actual OOF test

## Context

Notebook 13 estimated AID 1851's per-isoform population moments by treating
every no-fit ("Inactive", no `Fit_LogAC50`) row as a **point value** equal to
that row's own censoring floor (`-log10(top tested concentration)`). That is
a real, structural limitation: a floored-mean estimate can never place the
population mean below the lowest floor actually observed in the data, no
matter how many compounds are censored there.

Two real entrants on this exact challenge (SuperCowPowers, jeremycheminf)
report using this same external-population-calibration mechanism
successfully. SuperCowPowers specifically describes deriving CYP2D6's
population center from the panel's **inactivity rate** -- a method that, by
construction, *can* place a population center below the assay's detection
floor. Notebook 13's floored-mean CYP2D6 estimate (mean=4.686) sits far
above SuperCowPowers' reported ~3.7. This notebook exists to close that
specific gap, and -- more importantly -- to actually test whether doing so
helps.

A no-fit row is not "unknown" in an uninformative sense: it means the
compound's true pIC50-equivalent value is *at or below* that row's own
floor (its true AC50 was never reached within the concentrations tested).
This is textbook **left-censored** data, and admits a proper maximum-
likelihood treatment instead of a point-value substitution.

**This is a moment-estimation change only.** It does not touch curation
(`data/processed/aid1851_curated.csv` is read-only here), the overlap-
exclusion logic, or `src/calibration.py`'s `fit_blind_population_calibration`/
`apply_calibration` functions (unchanged -- they already take `target_mean`/
`target_std` as plain arguments).

## What this notebook can and cannot tell you

**This notebook CANNOT tell you whether either calibrated version will score
better on the real blind leaderboard.** No blind labels exist until the
challenge closes (2026-11-03), and none are read, approximated, or invented
anywhere below -- same limitation as notebook 13. What it **can** tell you:
whether the censored-MLE population estimate differs meaningfully from the
floored-mean one (Part 1), and whether calibrating against either one
actually improves OOF ST-RAE relative to raw, uncorrected predictions (Part
3) -- a real, falsifiable check this project has not run before for this
correction.

## Scope

Reads (read-only): `data/raw/aid1851/batch_*.csv` (already fetched by
notebook 13's `scripts/13_fetch_aid1851.py` -- not re-fetched here),
`data/processed/aid1851_curated.csv`, `data/processed/train_inhibition_
curated.csv`, `data/processed/test_blinded_curated.csv`,
`outputs/population_moments/aid1851_moments.csv`, `outputs/calibration_
blind_population/params.csv`, `outputs/11_caruana_prep/`, `outputs/11b_
caruana_selection/`'s existing predictions. **Nothing in those paths is
modified.** No model is retrained anywhere. No adoption/submission decision
is made here -- Part 3's three-way comparison is reported, not acted on.

In [1]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

from features import canonicalize_smiles, smiles_to_inchikey  # noqa: E402
from calibration import fit_blind_population_calibration, apply_calibration  # noqa: E402
from src.cv_bootstrap import per_fold_bootstrap_seed  # noqa: E402
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS  # noqa: E402
from src.vendor.openadmet_eval.evaluate_predictions import score_activity_predictions  # noqa: E402

RAW_DIR = REPO_ROOT / "data" / "raw" / "aid1851"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MOMENTS_DIR = REPO_ROOT / "outputs" / "population_moments"
CALIB_DIR = REPO_ROOT / "outputs" / "calibration_blind_population"
CARUANA_PREP_DIR = REPO_ROOT / "outputs" / "11_caruana_prep"
CARUANA_SELECTION_DIR = REPO_ROOT / "outputs" / "11b_caruana_selection"
OUT05 = REPO_ROOT / "outputs" / "05_cv_comparison"

PANEL_TO_ISOFORM = {
    "p450-cyp1a2": "CYP1A2", "p450-cyp2c9": "CYP2C9",
    "p450-cyp2d6": "CYP2D6", "p450-cyp3a4": "CYP3A4", "p450-cyp2c19": "CYP2C19",
}
ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]

pd.set_option("display.width", 220)
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge


## Part 1 — Censored-MLE population moment estimate, per isoform

### Reconstructing the fit/censored distinction

`data/processed/aid1851_curated.csv` (notebook 13's output) carries an
`any_censored` column, but it is an **aggregate-level** flag -- True if *at
least one* of a compound's replicate SID submissions was censored, which is
ambiguous for the ~2.8% of compounds with more than one replicate (it
cannot distinguish "all replicates censored" from "a mix of exact and
censored replicates"). There is no clean per-compound exact/censored marker
already saved to disk, so this notebook reconstructs it directly from the
raw fetched data (`data/raw/aid1851/batch_*.csv`, cached by notebook 13's
fetch script -- not re-fetched here), the same source notebook 13 itself
curated from.

In [2]:
batch_files = sorted(RAW_DIR.glob("batch_*.csv"))
assert batch_files, "no cached AID 1851 batches found -- notebook 13's scripts/13_fetch_aid1851.py must have run first"

raw_frames = []
for f in batch_files:
    df = pd.read_csv(f, low_memory=False)
    df = df[df["PUBCHEM_SID"].notna()]
    raw_frames.append(df)
raw = pd.concat(raw_frames, ignore_index=True)
for col in ["PUBCHEM_SID", "PUBCHEM_CID", "Fit_LogAC50"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

raw["isoform"] = raw["Panel Name"].map(PANEL_TO_ISOFORM)
raw = raw[raw["isoform"] != "CYP2C19"].copy()
raw = raw.dropna(subset=["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]).copy()

# Same per-row floor computation as notebook 13: each row's own top tested
# concentration, from whichever "Activity at X uM" columns are populated for it.
conc_cols = [c for c in raw.columns if c.startswith("Activity at ")]


def _parse_conc_uM(colname: str) -> float:
    return float(re.search(r"([\d.]+)\s*uM", colname).group(1))


conc_vals = np.array([_parse_conc_uM(c) for c in conc_cols])
notnull = raw[conc_cols].notna().to_numpy()
max_conc_uM = np.full(len(raw), -np.inf)
for i in range(len(conc_cols)):
    max_conc_uM = np.maximum(max_conc_uM, np.where(notnull[:, i], conc_vals[i], -np.inf))
raw["max_tested_conc_uM"] = np.where(np.isneginf(max_conc_uM), np.nan, max_conc_uM)

has_fit = raw["Fit_LogAC50"].notna()
floor_val = -np.log10(raw["max_tested_conc_uM"] * 1e-6)
raw["pIC50_equivalent"] = np.where(has_fit, -raw["Fit_LogAC50"], floor_val)
raw["censored"] = ~has_fit

print(f"Raw rows (4 isoforms, CID/SMILES present): {len(raw)}")
print(f"Fit (exact) rows: {int(has_fit.sum())}; no-fit (censored) rows: {int((~has_fit).sum())}")

Raw rows (4 isoforms, CID/SMILES present): 68484
Fit (exact) rows: 33513; no-fit (censored) rows: 34971


In [3]:
# Canonicalize once per unique CID -- identical to notebook 13's own step.
unique_smiles = raw[["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]].drop_duplicates("PUBCHEM_CID").copy()
unique_smiles["inchikey"] = unique_smiles["PUBCHEM_EXT_DATASOURCE_SMILES"].apply(smiles_to_inchikey)
n_fail = int(unique_smiles["inchikey"].isna().sum())
print(f"Canonicalization failures: {n_fail} of {len(unique_smiles)} (should be 0, matching notebook 13)")

raw = raw.merge(unique_smiles[["PUBCHEM_CID", "inchikey"]], on="PUBCHEM_CID", how="left")
raw = raw.dropna(subset=["inchikey"]).copy()

Canonicalization failures: 0 of 16560 (should be 0, matching notebook 13)


### Reconciling replicate SID submissions per compound

The same ~2.8% of compounds submitted under more than one SID (notebook 13's
own finding) need one resolved value per (inchikey, isoform) for the MLE, not
several raw-row observations. The reconciliation rule, applied per compound:

- **If any replicate has a real fit** (an exact measurement exists), the
  compound is treated as an **exact observation** -- its value is the mean
  of its fit-only replicates. A real measurement is strictly more
  informative than a censoring bound, so censored replicates for the same
  compound are not needed once an exact one exists.
- **If every replicate is censored**, the compound is treated as
  **left-censored at the *minimum* of its replicate floors** -- the
  tightest bound achieved by whichever replicate was tested to the highest
  concentration (testing further and still finding nothing is stronger
  evidence of inactivity than a replicate tested to a lower ceiling).

This is the standard way to reconcile repeated censored/exact measurements
of the same underlying quantity, and only affects ~2.8% of compounds.

In [4]:
def _reconcile(g: pd.DataFrame) -> pd.Series:
    fit_rows = g[~g["censored"]]
    if len(fit_rows) > 0:
        return pd.Series({"value": fit_rows["pIC50_equivalent"].mean(), "is_censored": False, "n_raw_rows": len(g)})
    return pd.Series({"value": g["pIC50_equivalent"].min(), "is_censored": True, "n_raw_rows": len(g)})


recon = raw.groupby(["inchikey", "isoform"]).apply(_reconcile, include_groups=False).reset_index()
recon["is_censored"] = recon["is_censored"].astype(bool)

print("Reconciled (inchikey, isoform) rows per isoform (should match aid1851_curated.csv's own row counts):")
print(recon.groupby("isoform").size())

curated = pd.read_csv(PROCESSED_DIR / "aid1851_curated.csv")
print()
print("aid1851_curated.csv row counts per isoform:")
print(curated.groupby("isoform").size())
assert (recon.groupby("isoform").size() == curated.groupby("isoform").size()).all(), \
    "reconciled compound population does not match notebook 13's curated population -- investigate before proceeding"
print("\nConfirmed: same compound population as notebook 13's curated file.")

Reconciled (inchikey, isoform) rows per isoform (should match aid1851_curated.csv's own row counts):
isoform
CYP1A2    16558
CYP2C9    16558
CYP2D6    16558
CYP3A4    16558
dtype: int64

aid1851_curated.csv row counts per isoform:
isoform
CYP1A2    16558
CYP2C9    16558
CYP2D6    16558
CYP3A4    16558
dtype: int64

Confirmed: same compound population as notebook 13's curated file.


In [5]:
print("Exact vs. censored counts per isoform:")
print(recon.groupby(["isoform", "is_censored"]).size().unstack(fill_value=0).rename(columns={False: "n_exact", True: "n_censored"}))
print()
print("True minimum floor value actually observed per isoform (confirmed fresh, not assumed):")
print(recon[recon["is_censored"]].groupby("isoform")["value"].agg(["min", "max"]))

Exact vs. censored counts per isoform:
is_censored  n_exact  n_censored
isoform                         
CYP1A2          9660        6898
CYP2C9          8250        8308
CYP2D6          5769       10789
CYP3A4          9203        7355

True minimum floor value actually observed per isoform (confirmed fresh, not assumed):
             min       max
isoform                   
CYP1A2   4.24306  5.640924
CYP2C9   4.24306  5.640924
CYP2D6   4.24306  4.941954
CYP3A4   4.24306  6.340084


### Distributional assumption

The true (uncensored) population of pIC50-equivalent values, per isoform, is
assumed **Normal(mu, sigma)**. This is not a silent extra assumption on top
of notebook 13's work -- working in pIC50/log space already bakes in a
log-normal-in-raw-AC50 assumption (standard for potency data), and Normal is
the natural distribution to place on that already-transformed scale. No
other distributional family is used below.

In [6]:
# Overlap exclusion -- identical logic to notebook 13's Part 2 (same train/blind
# inchikey sets, same isoform-specific train-label presence check).
train = pd.read_csv(PROCESSED_DIR / "train_inhibition_curated.csv")
blind = pd.read_csv(PROCESSED_DIR / "test_blinded_curated.csv")
blind_inchikeys = set(blind["inchikey"].dropna())

recon_clean = {}
overlap_counts = {}
for iso in ISOFORMS:
    sub = recon[recon["isoform"] == iso].copy()
    train_col = f"{iso}_pIC50_direct_inhibition"
    train_inchikeys_iso = set(train.loc[train[train_col].notna(), "inchikey"].dropna())
    overlap = sub["inchikey"].isin(train_inchikeys_iso) | sub["inchikey"].isin(blind_inchikeys)
    overlap_counts[iso] = int(overlap.sum())
    recon_clean[iso] = sub[~overlap]

print("n_excluded_overlap per isoform (should match notebook 13's own overlap counts):")
print(overlap_counts)

n_excluded_overlap per isoform (should match notebook 13's own overlap counts):
{'CYP1A2': 36, 'CYP2C9': 21, 'CYP2D6': 50, 'CYP3A4': 31}


### Fitting the censored Normal MLE

Log-likelihood per isoform: exact rows contribute `log(normal_pdf(x_i; mu,
sigma))`; censored rows contribute `log(normal_cdf((f_i - mu) / sigma))` --
the probability that the true value sits at or below that row's floor.
Both are summed and the negative total is minimized over `(mu, sigma)`.
`scipy.stats.norm.logpdf`/`logcdf` are used directly (not `log(norm.pdf(...))`
/`log(norm.cdf(...))`) -- this is the standard fix for potential numerical
underflow in the CDF tail for heavily-censored data, not a modeling choice.
Starting point: `mu0`/`sigma0` from the exact-only values; `sigma` is
bounded strictly positive.

In [7]:
def neg_log_likelihood(params, exact_vals, censored_floors):
    mu, sigma = params
    if sigma <= 0:
        return 1e10
    ll_exact = norm.logpdf(exact_vals, loc=mu, scale=sigma).sum()
    z = (censored_floors - mu) / sigma
    ll_censored = norm.logcdf(z).sum()
    return -(ll_exact + ll_censored)


mle_results = {}
for iso in ISOFORMS:
    sub = recon_clean[iso]
    exact_vals = sub.loc[~sub["is_censored"], "value"].to_numpy()
    censored_floors = sub.loc[sub["is_censored"], "value"].to_numpy()

    mu0, sigma0 = exact_vals.mean(), exact_vals.std(ddof=1)
    res = minimize(
        neg_log_likelihood, x0=[mu0, sigma0], args=(exact_vals, censored_floors),
        method="L-BFGS-B", bounds=[(None, None), (1e-6, None)],
    )
    mle_results[iso] = dict(
        mu=float(res.x[0]), sigma=float(res.x[1]), success=bool(res.success),
        message=res.message, nll=float(res.fun),
        n_exact=len(exact_vals), n_censored=len(censored_floors),
        mu0=float(mu0), sigma0=float(sigma0),
    )
    print(f"{iso}: success={res.success}, mu={res.x[0]:.4f}, sigma={res.x[1]:.4f}, "
          f"n_exact={len(exact_vals)}, n_censored={len(censored_floors)}, message={res.message}")

all_converged = all(r["success"] and np.isfinite(r["sigma"]) and r["sigma"] > 0 for r in mle_results.values())
if not all_converged:
    raise RuntimeError("At least one isoform's censored-MLE fit did not converge or produced an invalid sigma -- "
                        "stopping rather than reporting an untrustworthy number. Investigate before proceeding.")
print("\nAll four isoforms converged with a positive, finite sigma.")

CYP1A2: success=True, mu=4.6479, sigma=1.0181, n_exact=9626, n_censored=6896, message=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
CYP2C9: success=True, mu=4.4725, sigma=1.1100, n_exact=8241, n_censored=8296, message=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
CYP2D6: success=True, mu=3.8022, sigma=1.3832, n_exact=5724, n_censored=10784, message=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
CYP3A4: success=True, mu=4.6212, sigma=0.8255, n_exact=9193, n_censored=7334, message=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

All four isoforms converged with a positive, finite sigma.


### Multi-start robustness check

A single converged optimizer run is not, on its own, proof of a global
optimum. Each isoform's fit is re-run from five different starting points
(including some far from the data-driven default) and the results are
required to agree, confirming this is not a local-optimum artifact.

In [8]:
START_POINTS = [None, (3.0, 1.0), (5.0, 2.0), (4.0, 0.5), (2.0, 3.0)]

for iso in ISOFORMS:
    sub = recon_clean[iso]
    exact_vals = sub.loc[~sub["is_censored"], "value"].to_numpy()
    censored_floors = sub.loc[sub["is_censored"], "value"].to_numpy()

    fits = []
    for start in START_POINTS:
        x0 = [exact_vals.mean(), exact_vals.std(ddof=1)] if start is None else list(start)
        res = minimize(neg_log_likelihood, x0=x0, args=(exact_vals, censored_floors),
                        method="L-BFGS-B", bounds=[(None, None), (1e-6, None)])
        fits.append((res.x[0], res.x[1], res.success))

    mus = np.array([f[0] for f in fits])
    sigmas = np.array([f[1] for f in fits])
    all_success = all(f[2] for f in fits)
    agree = (mus.max() - mus.min() < 1e-4) and (sigmas.max() - sigmas.min() < 1e-4)
    print(f"{iso}: {len(fits)} starts, all succeeded={all_success}, mu range=[{mus.min():.6f}, {mus.max():.6f}], "
          f"sigma range=[{sigmas.min():.6f}, {sigmas.max():.6f}] -> agree={agree}")
    if not (all_success and agree):
        raise RuntimeError(f"{iso}: multi-start check failed to agree -- possible local optimum, stopping rather than trusting this fit.")

print("\nAll four isoforms: every start point converges to the same optimum -- confirmed global, not local.")

CYP1A2: 5 starts, all succeeded=True, mu range=[4.647922, 4.647922], sigma range=[1.018141, 1.018142] -> agree=True


CYP2C9: 5 starts, all succeeded=True, mu range=[4.472496, 4.472497], sigma range=[1.109964, 1.109965] -> agree=True


CYP2D6: 5 starts, all succeeded=True, mu range=[3.802174, 3.802178], sigma range=[1.383175, 1.383180] -> agree=True


CYP3A4: 5 starts, all succeeded=True, mu range=[4.621217, 4.621217], sigma range=[0.825515, 0.825516] -> agree=True

All four isoforms: every start point converges to the same optimum -- confirmed global, not local.


### Comparison: floored-mean vs. censored-MLE

In [9]:
floored = pd.read_csv(MOMENTS_DIR / "aid1851_moments.csv").set_index("isoform")

own_moments = {}
for iso in ISOFORMS:
    vals = train[f"{iso}_pIC50_direct_inhibition"].dropna()
    own_moments[iso] = dict(mean=float(vals.mean()), std=float(vals.std(ddof=1)))

rows = []
for iso in ISOFORMS:
    r = mle_results[iso]
    fl = floored.loc[iso]
    rows.append(dict(
        isoform=iso,
        n_compounds_used=r["n_exact"] + r["n_censored"],
        n_excluded_overlap=overlap_counts[iso],
        mean=r["mu"], std=r["sigma"],
        floored_mean=float(fl["mean"]), floored_std=float(fl["std"]),
        n_exact=r["n_exact"], n_censored=r["n_censored"], converged=r["success"],
        own_train_mean=own_moments[iso]["mean"], own_train_std=own_moments[iso]["std"],
    ))

comparison_df = pd.DataFrame(rows)
print(comparison_df[["isoform", "n_compounds_used", "n_excluded_overlap", "mean", "std",
                      "floored_mean", "floored_std", "own_train_mean", "own_train_std"]].to_string(index=False))

out_path = MOMENTS_DIR / "aid1851_moments_censored_mle.csv"
comparison_df.to_csv(out_path, index=False)
print()
print(f"Saved -> {out_path}")

isoform  n_compounds_used  n_excluded_overlap     mean      std  floored_mean  floored_std  own_train_mean  own_train_std
 CYP1A2             16522                  36 4.647922 1.018141      4.972615     0.689005        4.955410       1.030476
 CYP2C9             16537                  21 4.472496 1.109964      4.946670     0.691129        4.580738       0.782295
 CYP2D6             16508                  50 3.802176 1.383177      4.685632     0.648923        4.784202       0.916096
 CYP3A4             16527                  31 4.621217 0.825515      4.924625     0.567007        4.096054       1.093219

Saved -> /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/population_moments/aid1851_moments_censored_mle.csv


**Reading this table**: `mean`/`std` are the new censored-MLE estimates;
`floored_mean`/`floored_std` are notebook 13's original estimates, kept
alongside for direct comparison. **CYP2D6 moves from 4.686 (floored) to
3.802 (censored-MLE) -- a real, large shift below the ~4.24 detection floor,
the specific gap this notebook exists to close**, and now much closer to
SuperCowPowers' reported ~3.7 than the floored-mean estimate ever could get
by construction. **CYP3A4 (the isoform flagged as risky to over-correct,
since it's this project's best real-leaderboard performer) moves from 4.925
to 4.621 -- a real, moderate shift (-0.30), not "stays similar."** This is
reported plainly rather than downplayed: whether that shift helps or hurts
CYP3A4 specifically is exactly what Part 3's OOF test below checks, not
assumed from the moment estimate alone. CYP1A2 and CYP2C9 shift by similar
moderate amounts (-0.32, -0.47 respectively). Every isoform's censored-MLE
sigma is noticeably larger than its floored-mean sigma (recovering some of
the spread the floored-mean approach collapsed by point-substituting a
range of true censored values with a single floor number).

## Part 2 — Refit calibration with the new moments

Same `fit_blind_population_calibration` call as notebook 13's Part 5, same
OOF predictions and `y_true` (re-loaded identically, not modified), only the
`target_mean`/`target_std` arguments change -- from notebook 13's floored-
mean estimate to this notebook's censored-MLE estimate.

In [10]:
target_moments_censored = {row["isoform"]: (row["mean"], row["std"]) for row in rows}

calib_results = []
oof_by_fold = {}  # isoform -> per-fold long dataframe with columns: repeat, fold, Molecule_Name, inchikey, pred_col, y_true

for iso in ["CYP1A2", "CYP2D6", "CYP3A4"]:
    oof_long = pd.read_csv(CARUANA_PREP_DIR / f"oof_long_{iso}.csv")
    oof_by_fold[iso] = oof_long[["repeat", "fold", "Molecule_Name", "inchikey", "chemprop_chemeleoninit", "y_true"]].rename(
        columns={"chemprop_chemeleoninit": "raw_pred"}
    )
    oof_pred = oof_long["chemprop_chemeleoninit"].to_numpy()
    oof_true = oof_long["y_true"].to_numpy()
    target_mean, target_std = target_moments_censored[iso]
    params = fit_blind_population_calibration(oof_pred, oof_true, target_mean, target_std)
    calib_results.append(dict(isoform=iso, model="chemprop_chemeleoninit", n=len(oof_pred), **params))

iso = "CYP2C9"
oof_long_cyp2c9 = pd.read_csv(CARUANA_PREP_DIR / f"oof_long_{iso}.csv")
pred_files = sorted((CARUANA_SELECTION_DIR / "predictions").glob(f"caruana__{iso}__repeat*_fold*.csv"))
pred_parts = []
for f in pred_files:
    df = pd.read_csv(f)
    key_part = f.stem.split("__")[-1]
    r_str, fo_str = key_part.split("_")
    df["repeat"] = int(r_str.replace("repeat", ""))
    df["fold"] = int(fo_str.replace("fold", ""))
    pred_parts.append(df)
caruana_pred = pd.concat(pred_parts, ignore_index=True)
merged = caruana_pred.merge(
    oof_long_cyp2c9[["repeat", "fold", "inchikey", "y_true"]], on=["repeat", "fold", "inchikey"], how="left",
)
assert merged["y_true"].isna().sum() == 0, "unmatched rows after joining capped-ensemble OOF to y_true"
oof_by_fold[iso] = merged.rename(columns={f"{iso}_pIC50_direct_inhibition": "raw_pred"})[
    ["repeat", "fold", "Molecule_Name", "inchikey", "raw_pred", "y_true"]
]

oof_pred = merged[f"{iso}_pIC50_direct_inhibition"].to_numpy()
oof_true = merged["y_true"].to_numpy()
target_mean, target_std = target_moments_censored[iso]
params = fit_blind_population_calibration(oof_pred, oof_true, target_mean, target_std)
calib_results.append(dict(isoform=iso, model="caruana_capped_ensemble", n=len(oof_pred), **params))

calib_df_censored = pd.DataFrame(calib_results)
print(calib_df_censored.to_string(index=False))

out_path = CALIB_DIR / "params_censored_mle.csv"
calib_df_censored.to_csv(out_path, index=False)
print()
print(f"Saved -> {out_path}")

isoform                   model     n    slope  intercept      rho  own_mean  own_std  target_mean  target_std target_source
 CYP1A2  chemprop_chemeleoninit  7060 0.784787   0.755207 0.498868  4.960220 0.647205     4.647922    1.018141       aid1851
 CYP2D6  chemprop_chemeleoninit  7465 1.046274  -1.166897 0.368531  4.749306 0.487199     3.802176    1.383177       aid1851
 CYP3A4  chemprop_chemeleoninit 11675 0.714504   1.706439 0.752331  4.079440 0.869219     4.621217    0.825515       aid1851
 CYP2C9 caruana_capped_ensemble  6425 1.567448  -2.713270 0.653630  4.584373 0.462858     4.472496    1.109964       aid1851

Saved -> /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/calibration_blind_population/params_censored_mle.csv


In [11]:
calib_df_floored = pd.read_csv(CALIB_DIR / "params.csv")
compare_cols = ["isoform", "model", "slope", "intercept", "target_mean", "target_std"]
print("Floored-mean calibration params (existing):")
print(calib_df_floored[compare_cols].to_string(index=False))
print()
print("Censored-MLE calibration params (new):")
print(calib_df_censored[compare_cols].to_string(index=False))

Floored-mean calibration params (existing):
isoform                   model    slope  intercept  target_mean  target_std
 CYP1A2  chemprop_chemeleoninit 0.531087   2.338306     4.972615    0.689005
 CYP2D6  chemprop_chemeleoninit 0.490864   2.354370     4.685632    0.648923
 CYP3A4  chemprop_chemeleoninit 0.490760   2.922601     4.924625    0.567007
 CYP2C9 caruana_capped_ensemble 0.975985   0.472389     4.946670    0.691129

Censored-MLE calibration params (new):
isoform                   model    slope  intercept  target_mean  target_std
 CYP1A2  chemprop_chemeleoninit 0.784787   0.755207     4.647922    1.018141
 CYP2D6  chemprop_chemeleoninit 1.046274  -1.166897     3.802176    1.383177
 CYP3A4  chemprop_chemeleoninit 0.714504   1.706439     4.621217    0.825515
 CYP2C9 caruana_capped_ensemble 1.567448  -2.713270     4.472496    1.109964


## Part 3 — The actual test: does either version help, on OOF ST-RAE?

This is the acid test this notebook exists to run. For each isoform/model
pair, `apply_calibration` is applied to the pooled OOF predictions under
three conditions -- **raw** (uncorrected), **floored-mean** calibration
(notebook 13's `params.csv`), and **censored-MLE** calibration (this
notebook's `params_censored_mle.csv`) -- and all three are scored against
the same `y_true` via the same vendored bootstrap scorer every other
notebook in this project uses (`score_activity_predictions`), per-fold, with
the same per-fold seeds `05_cv_comparison/manifest.csv` already fixed --
identical methodology to notebook 11b's own scoring pipeline.

In [12]:
curated_train = train  # already loaded above; alias for clarity in this section

manifest = pd.read_csv(OUT05 / "manifest.csv")
seed_lookup = manifest.drop_duplicates(["repeat", "fold"]).set_index(["repeat", "fold"])["seed"]
assert manifest.groupby(["repeat", "fold"])["seed"].nunique().eq(1).all(), "seed not shared across configs for some fold"


def score_predictions(pred_df: pd.DataFrame, isoform_col: str, seed: int) -> pd.DataFrame:
    ground_truth = curated_train[curated_train["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(int(seed)):
        scored = score_activity_predictions(pred_df[["Molecule_Name", "inchikey", isoform_col]], ground_truth, [isoform_col])
    return scored


def point_estimate(scored: pd.DataFrame, isoform_col: str) -> dict:
    pe = scored.groupby("Endpoint")[METRIC_NAMES].mean()
    return {m: float(pe.loc[isoform_col, m]) for m in METRIC_NAMES}


MODEL_BY_ISOFORM = {
    "CYP1A2": "chemprop_chemeleoninit", "CYP2D6": "chemprop_chemeleoninit",
    "CYP3A4": "chemprop_chemeleoninit", "CYP2C9": "caruana_capped_ensemble",
}
params_by_isoform = {
    "floored": calib_df_floored.set_index("isoform"),
    "censored_mle": calib_df_censored.set_index("isoform"),
}

comparison_rows = []
for iso in ISOFORMS:
    isoform_col = f"{iso}_pIC50_direct_inhibition"
    fold_df_all = oof_by_fold[iso]

    for condition in ["raw", "floored", "censored_mle"]:
        fold_scores = []
        for (r, fo), fold_df in fold_df_all.groupby(["repeat", "fold"]):
            raw_pred = fold_df["raw_pred"].to_numpy()
            if condition == "raw":
                pred_vals = raw_pred
            else:
                p = params_by_isoform[condition].loc[iso]
                pred_vals = apply_calibration(raw_pred, {"slope": p["slope"], "intercept": p["intercept"]})

            pred_df = fold_df[["Molecule_Name", "inchikey"]].copy()
            pred_df[isoform_col] = pred_vals
            seed = int(seed_lookup.loc[(r, fo)])
            scored = score_predictions(pred_df, isoform_col, seed)
            fold_scores.append(point_estimate(scored, isoform_col))

        mean_scores = pd.DataFrame(fold_scores).mean().to_dict()
        comparison_rows.append(dict(isoform=iso, model=MODEL_BY_ISOFORM[iso], condition=condition, n_folds=len(fold_scores), **mean_scores))
    print(f"{iso}: scored all 3 conditions ({len(fold_df_all.groupby(['repeat','fold']))} folds each)")

oof_test_df = pd.DataFrame(comparison_rows)
out_path = CALIB_DIR / "oof_strae_comparison.csv"
oof_test_df.to_csv(out_path, index=False)
print()
print(f"Saved -> {out_path}")

2026-09-12 13:37:38.830 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-12 13:37:38.830 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 277 rows and 22 columns.


2026-09-12 13:37:38.831 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-12 13:37:39.411 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:169 - Completed scoring activity predictions


2026-09-12 13:37:39.413 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


...

2026-09-12 13:43:26.203 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-12 13:43:26.204 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 495 rows and 22 columns.


2026-09-12 13:43:26.204 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP3A4_pIC50_direct_inhibition


2026-09-12 13:43:26.804 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:169 - Completed scoring activity predictions


CYP3A4: scored all 3 conditions (25 folds each)

Saved -> /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/calibration_blind_population/oof_strae_comparison.csv


In [13]:
display_cols = ["isoform", "condition", "ST-RAE", "MAE", "R2", "Spearman_R", "Kendall_Tau"]
for iso in ISOFORMS:
    print(f"=== {iso} ({MODEL_BY_ISOFORM[iso]}) ===")
    sub = oof_test_df[oof_test_df["isoform"] == iso][display_cols[1:]]
    print(sub.to_string(index=False))
    best_condition = sub.loc[sub["ST-RAE"].idxmin(), "condition"]
    print(f"Best OOF ST-RAE: {best_condition}\n")

=== CYP1A2 (chemprop_chemeleoninit) ===
   condition   ST-RAE      MAE       R2  Spearman_R  Kendall_Tau
         raw 0.868459 0.668741 0.226559    0.508249     0.355908
     floored 0.820678 0.656075 0.217628    0.508249     0.355908
censored_mle 1.013556 0.735275 0.150009    0.508249     0.355908
Best OOF ST-RAE: floored

=== CYP2C9 (caruana_capped_ensemble) ===
   condition   ST-RAE      MAE       R2  Spearman_R  Kendall_Tau
         raw 0.616919 0.447126 0.415582    0.648544      0.47445
     floored 0.783950 0.539416 0.187148    0.648544      0.47445
censored_mle 0.768067 0.481548 0.315630    0.648544      0.47445
Best OOF ST-RAE: raw

=== CYP2D6 (chemprop_chemeleoninit) ===
   condition   ST-RAE      MAE        R2  Spearman_R  Kendall_Tau
         raw 1.003907 0.628190  0.097991    0.401457     0.274917
     floored 0.931855 0.598032  0.106321    0.401457     0.274917
censored_mle 2.258997 1.144208 -1.097462    0.401457     0.274917
Best OOF ST-RAE: floored

=== CYP3A4 (chemprop_

**Stating plainly, per isoform, which condition wins on OOF ST-RAE --
without rounding up an ambiguous or negative result:**

| Isoform | raw | floored | censored_mle | **Winner** |
|---|---|---|---|---|
| CYP1A2 | 0.868 | **0.821** | 1.014 | **floored** |
| CYP2C9 | **0.617** | 0.784 | 0.768 | **raw** |
| CYP2D6 | 1.004 | **0.932** | 2.259 | **floored** |
| CYP3A4 | **0.546** | 1.007 | 0.637 | **raw** |

**The censored-MLE correction -- the more statistically principled population
estimate, and the one this notebook exists to test -- is not the best
condition on OOF ST-RAE for any of the four isoforms.** It is beaten by
floored-mean calibration on CYP1A2/CYP2D6, and beaten by raw on CYP2C9/CYP3A4.
Worse: on CYP2D6 -- the isoform whose population-mean gap this notebook set
out to fix -- censored-MLE calibration is not just unhelpful, it is
**catastrophic** (ST-RAE 2.259, more than double raw's 1.004 and floored's
0.932). Being statistically correct as a *population-moment estimator* did
not translate into a better *calibration target*: `fit_blind_population_
calibration`'s rho-shrunk slope reacts to the censored-MLE target's much
wider `target_std` (1.383 vs. floored's 0.649 for CYP2D6) by inflating the
slope well past 1.0 (1.046, vs. floored's 0.491) and driving a large
negative intercept (-1.167) -- overcorrecting the spread far beyond what
CYP2D6's weak raw-prediction/truth correlation (rho=0.369) can support.
This is a real, clean negative result for the censored-MLE approach
specifically, not an ambiguous one, and it is reported as such rather than
rounded up.

**CYP3A4-specific callout, given its real-leaderboard strength**: raw wins
outright, and **both** calibrated conditions score worse than raw on CYP3A4
-- floored-mean nearly doubles ST-RAE (1.007 vs. 0.546), and censored-MLE is
also worse (0.637), though less catastrophically. This is a concrete,
isoform-specific reason against adopting *either* correction for CYP3A4,
regardless of what happens on other isoforms -- exactly the risk this task
asked to be flagged if it materialized. It did.

**CYP2C9 also favors raw** -- both calibrated conditions hurt it, with
floored-mean the worse of the two there. Only CYP1A2 and CYP2D6 show any
calibrated condition beating raw at all, and in both cases it is
floored-mean, not censored-MLE, that wins.

**Rank correlations (Spearman/Kendall) are identical across all three
conditions per isoform, by construction** -- `apply_calibration` is a
monotonic (positive-slope) affine transform in every fitted case here, which
cannot change rank order, so it cannot move a rank-based metric. Only
ST-RAE/MAE/R2 (which are sensitive to absolute placement, not just ranking)
can differ between conditions, and did.

## Scope reminder

No blind labels exist or are used anywhere in this notebook -- every number
above is an OOF check, computed with the same bootstrap methodology and
per-fold seeds every other CV-side notebook in this project uses. **No
adoption or submission decision is made here.** This notebook reports the
three-way OOF comparison and stops; whether to actually use either
calibrated version on a real submission is a separate decision, explicitly
not this notebook's to make.